In [ ]:
import ee
import geemap

In [ ]:
# === CONFIGURATION ===
# Each user sets their own GEE project here.
# (Once the notebook is integrated into the repo, this will be read from config/config.yaml)
GEE_PROJECT = 'alert-result-474012-g5'   # <-- replace with your own GEE project

# === CONNECTION ===
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

print("GEE connected successfully")

In [ ]:
# === STUDY AREA — Chapada dos Veadeiros + 25 km buffer ===
# Python translation of scripts/01_define_aoi.js
# Expected area: ~11,934.66 km² (control number)

# Load protected-area polygons (WDPA)
wdpa = ee.FeatureCollection('WCMC/WDPA/current/polygons')

# Auxiliary rectangle, only to clip to Goias and exclude PARNA das Emas [W, S, E, N]
region_goias = ee.Geometry.Rectangle([-48.5, -14.8, -46.8, -13.0])

# Filter by name and bounds
chapada = (wdpa
           .filter(ee.Filter.stringContains('NAME', 'Cerrado Protected'))
           .filterBounds(region_goias))

# Build the AOI: clip to Goias + 25 km buffer (in meters)
aoi = (chapada.geometry()
       .intersection(region_goias, 100)   # clip to Goias (maxError = 100 m)
       .buffer(25000))                     # 25 km buffer, in meters

# --- Reproducibility control ---
area_km2 = aoi.area(100).divide(1e6).getInfo()
print(f"AOI area: {area_km2:.2f} km²")
print("Expected: ~11934.66 km²")

if 11000 < area_km2 < 13000:
    print("Area is correct")
elif area_km2 < 3000:
    print("Too small — buffer probably missing")
elif area_km2 > 100000:
    print("Too large — grabbed the rectangular bbox by mistake")

In [ ]:
# Install geemap (only needed once per Colab session)
!pip install geemap -q
print("geemap installed ")

In [ ]:
# Clip the park polygon to Goias only (for the reference layer)
chapada_goias = chapada.geometry().intersection(region_goias, 100)

# Create the map
Map = geemap.Map()
Map.centerObject(aoi, 9)
Map.add_basemap('SATELLITE')

# AOI (park + 25 km buffer) — your working area
Map.addLayer(aoi, {'color': 'red'}, 'AOI (Chapada + 25 km buffer)')

# Park boundary clipped to Goias (reference) — no more Emas
Map.addLayer(chapada_goias, {'color': 'blue'}, 'Park boundary (Chapada only)')

Map

In [ ]:
# === LAND-COVER MASK (MapBiomas Collection 9) ===
# Keep only native Cerrado vegetation; exclude agriculture, pasture, urban, water.

# MapBiomas Collection 9 asset (annual land-cover, 1985-2023)
mapbiomas = ee.Image('projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_integration_v1')

# Select the 2023 band (most recent year)
lulc_2023 = mapbiomas.select('classification_2023')

# Native vegetation classes to KEEP (from config):
# 3=Forest Formation, 4=Savanna Formation, 11=Wetland,
# 12=Grassland Formation, 29=Rocky Outcrop
keep_classes = [3, 4, 11, 12, 29]

# Build the native-vegetation mask (1 = native, 0 = anthropic/water)
native_mask = lulc_2023.remap(
    keep_classes,                 # from these classes
    [1] * len(keep_classes),      # all mapped to 1
    0                             # everything else -> 0
)

# Clip to our AOI
native_mask = native_mask.clip(aoi)

print("Native-vegetation mask created")

In [ ]:
# === NATIVE-VEGETATION MASK ===
# Keep only native Cerrado classes: 3, 4, 11, 12, 29

keep_classes = [3, 4, 11, 12, 29]

# Build the mask: 1 = native vegetation, 0 = everything else
native_mask = lulc_2023.remap(
    keep_classes,                 # from these classes
    [1] * len(keep_classes),      # all mapped to 1
    0                             # everything else -> 0
).clip(aoi)

print("Native-vegetation mask created")

In [ ]:
# === VISUALIZE the native-vegetation mask ===

Map = geemap.Map()
Map.centerObject(aoi, 9)
Map.add_basemap('SATELLITE')

# Show the full MapBiomas land cover (all classes, with official colors)
Map.addLayer(
    lulc_2023,
    {'min': 0, 'max': 62, 'palette': ['ffffff']},  # placeholder palette
    'MapBiomas 2023 (all classes)',
    False  # start hidden
)

# Show the native-vegetation mask (green = native, kept)
Map.addLayer(
    native_mask.selfMask(),   # selfMask hides the zeros
    {'palette': ['0000FF']},
    'Native vegetation (kept)'
)

# AOI outline for reference
Map.addLayer(aoi, {'color': 'red'}, 'AOI outline')

Map